# CSTR — Raw Construction

This notebook extends `Example2_batch_fermenter` by adding continuous liquid
inlet and outlet boundaries, converting the batch vessel into a continuously
stirred tank reactor (CSTR). Every layer below the two new boundaries is
identical to Example2 — read that notebook first if you have not already.

| Layer | Example2 (batch) | Example3 (CSTR) |
|---|---|---|
| Species + reactions | as-is | unchanged |
| Gas sparging + venting | as-is | unchanged |
| pH controller | as-is | unchanged |
| Liquid inlet | — | **`LiquidFeed`** |
| Liquid outlet | — | **`LiquidDrain`** |

## CSTR steady state

At steady state the net specific growth rate equals the dilution rate:

$$
\mu^* = D = \frac{Q}{V_L}
$$

For Monod kinetics this uniquely sets the residual substrate concentration:

$$
S^* = \frac{K_S \, D}{\mu_{\max} - D}
$$

and the steady-state biomass follows from the substrate balance:

$$
X^* = Y (S_0 - S^*)
$$

where $S_0$ is the feed substrate concentration and $Y$ is the yield
coefficient. These theoretical values are computed analytically in
Section 14 and compared against the simulation result.

## 1 · Environment setup

In [ ]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

_root = _find_root()
if str(_root / "models") not in sys.path:
    sys.path.insert(0, str(_root / "models"))

import numpy as np
import matplotlib.pyplot as plt

from PyOMES.chemistry import Species
from PyOMES.chemistry.databases.anaerobic_digestion import AD_BASIC
from PyOMES.reactions import (
    EquilibriumReaction, KineticReaction,
    ReactionBuilder, ReactionSystem,
)
from PyOMES.core import (
    ControlVolume, EquilibriumTransferModel, GasPhase, KineticTransferModel,
    LiquidPhase, Simulation,
)
from PyOMES.core.boundaries import (
    GasFeed, LiquidDrain, LiquidFeed, PressureReliefVent,
)
from PyOMES.core.phases import R_L_ATM_MOL_K
from PyOMES.control.cv_loops import PHController

print("Imports OK - repo root:", _root)

## 2 · Physical parameters

The vessel geometry and sparging parameters are identical to Example2. The
three CSTR-specific parameters introduced here are `D_PER_H` (dilution rate),
`S_FEED_G_L` (feed substrate concentration), and `Q_L_per_h` (volumetric flow
rate), which is derived from the first two.

**Dilution rate** $D = Q / V_L$ has units of 1/h and is the inverse of the
hydraulic retention time (HRT). Choosing $D < \mu_{\max}$ ensures the reactor
operates below washout; choosing $D$ close to $\mu_{\max}$ gives a low residual
substrate but high washout risk. Here $D = 0.2$ /h (HRT = 5 h) is a
conservative operating point well below $\mu_{\max} = 0.5$ /h.

In [ ]:
# Vessel geometry — same as Example2
T_K            = 305.15   # reactor temperature (K)
V_TOTAL_L      = 2.0      # total vessel volume (L)
HEADSPACE_FRAC = 0.20     # fraction of volume that is headspace
V_GAS = V_TOTAL_L * HEADSPACE_FRAC
V_LIQ = V_TOTAL_L * (1.0 - HEADSPACE_FRAC)

# Sparging and pH — same as Example2
KLA_PER_H   = {"O2": 150.0, "CO2": 135.0}
PH_SETPOINT = 5.0

# CSTR operating point
D_PER_H     = 0.2         # dilution rate (1/h); HRT = 1/D = 5 h
S_FEED_G_L  = 5.0         # feed substrate concentration (g/L)
Q_L_PER_H   = D_PER_H * V_LIQ   # volumetric flow rate (L/h) for constant volume

TAU_H   = 10.0   # run duration (h) — 2 HRTs gives near-steady-state
N_STEPS = 1000

print(f"V_gas = {V_GAS:.3f} L   V_liq = {V_LIQ:.3f} L   T = {T_K:.2f} K")
print(f"D     = {D_PER_H:.2f} /h   HRT = {1/D_PER_H:.1f} h")
print(f"Q     = {Q_L_PER_H:.4f} L/h   S_feed = {S_FEED_G_L:.1f} g/L")

## 3 · Species declarations

Identical to Example2. `YEAST` carries an explicit `MW` because the empirical
biomass formula (CH₁.₆₁O₀.₅₆) omits nitrogen.

In [ ]:
ACETIC_ACID   = Species(id="AceticAcid",  atoms={"C":2,"H":4,"O":2},       charge=0)
ACETATE_MINUS = Species(id="Acetate-",    atoms={"C":2,"H":3,"O":2},       charge=-1)
CO2           = Species(id="CO2",         atoms={"C":1,"O":2},              charge=0)
YEAST         = Species(id="Yeast",       atoms={"C":1,"H":1.61,"O":0.56}, charge=0, MW=24.626)

print("Locally declared species:")
for sp in [ACETIC_ACID, ACETATE_MINUS, CO2, YEAST]:
    print(f"  {sp.id:>14}  MW={float(sp.MW):.3f}  charge={sp.charge}")

## 4 · Kinetic reaction: aerobic growth on acetic acid

Identical to Example2. See that notebook for a full description of the Monod
rate law and the stoichiometry derivation.

In [ ]:
MU_MAX = 0.5    # 1/h
KS_G_L = 5e-3  # g/L
YIELD  = 0.36   # g biomass / g substrate

rxn_growth = ReactionBuilder.monod_aerobic_growth(
    substrate    = ACETIC_ACID,
    biomass      = YEAST,
    mu_max_per_h = MU_MAX,
    Ks_gL        = KS_G_L,
    yield_gX_gS  = YIELD,
    label        = "growth_on_AceticAcid",
)

print("Kinetic reaction:", rxn_growth.label)
print("Stoichiometry:")
for e in rxn_growth.stoichiometry:
    print(f"  {e.coefficient:+.4g}  {e.species.id}  ({e.phase})")

## 5 · Equilibrium reactions

Identical to Example2: water dissociation (5a), carbonate ladder (5b), acetic
acid dissociation (5c).

In [ ]:
rxn_water = EquilibriumReaction(
    "H2O,aq <-> H+,aq + OH-,aq",
    log_K=-14.0, dH_J_per_mol=55900.0, T_ref_K=298.15,
    label="eq_water",
)
print(rxn_water.label, f"  log_K = {rxn_water.log_K}")

In [ ]:
rxn_co2_aq = EquilibriumReaction(
    "CO2,aq + H2O,aq <-> HCO3-,aq + H+,aq",
    log_K=-6.35, dH_J_per_mol=7646.0, T_ref_K=298.15,
    total_id="CO2", label="eq_CO2",
)
rxn_co3 = EquilibriumReaction(
    "HCO3-,aq <-> H+,aq + CO3--,aq",
    log_K=-10.33, label="eq_HCO3",
)
print(rxn_co2_aq.label, f"  log_K = {rxn_co2_aq.log_K}")
print(rxn_co3.label,    f"  log_K = {rxn_co3.log_K}")

In [ ]:
rxn_dissoc = EquilibriumReaction(
    "AceticAcid,aq <-> Acetate-,aq + H+,aq",
    species={"AceticAcid": ACETIC_ACID, "Acetate-": ACETATE_MINUS},
    log_K=-4.756, label="eq_AceticAcid",
)
print(rxn_dissoc.label, f"  log_K = {rxn_dissoc.log_K}")

## 6 · Assemble the ReactionSystem

Identical to Example2.

In [ ]:
rxn_system = ReactionSystem(
    [rxn_growth, rxn_water, rxn_co2_aq, rxn_dissoc, rxn_co3],
    label="cstr_chemistry",
)

print(f"Kinetic reactions      : {len(rxn_system.kinetic_reactions)}")
print(f"Single-phase equilibria: {len(rxn_system.single_phase_equilibria)}")
print(f"  -> ", [r.label for r in rxn_system.single_phase_equilibria])
print(f"Species in system      : {', '.join(rxn_system.species_ids)}")

## 7 · Build the gas phase

Identical to Example2. The headspace is seeded with air at 1 atm.

In [ ]:
n_total_gas = (1.0 * V_GAS) / (R_L_ATM_MOL_K * T_K)

gas = GasPhase(
    n_mol={
        "O2":  n_total_gas * 0.2095,
        "CO2": n_total_gas * 0.0004,
        "N2":  n_total_gas * 0.7901,
    },
    V_L=V_GAS,
    T_K=T_K,
)

print(f"Total gas moles: {n_total_gas:.4f}")
for sp, n in gas.n_mol.items():
    print(f"  {sp:>4}: {n:.4e} mol")

## 8 · Build the liquid phase

The initial state is the same batch-fermenter seed used in Example2 (1.2 g/L
substrate, 0.1 g/L inoculum). From $t = 0$ the feed continuously replaces this
initial liquid; within one HRT (5 h) the reactor inventory is dominated by the
fed state rather than the initial charge.

In [ ]:
p_atm = gas.p_atm

n_acetate = (1.2 / float(ACETIC_ACID.MW)) * V_LIQ   # initial 1.2 g/L
n_yeast   = (0.1 / float(YEAST.MW))       * V_LIQ   # initial 0.1 g/L

def _henry_n(sp):
    pm = AD_BASIC.partition_models[sp]
    return pm.H_ref * p_atm.get(sp, 0.0) * 101325 / 1000 * V_LIQ

liquid = LiquidPhase(
    n_mol={
        ACETIC_ACID.id:   n_acetate,
        ACETATE_MINUS.id: 0.0,
        YEAST.id:         n_yeast,
        "O2":             _henry_n("O2"),
        "CO2":            _henry_n("CO2"),
        "N2":             _henry_n("N2"),
        "HCO3-":          0.0,
        "CO3--":          0.0,
        "OH-":            0.0,
        "H+":             1.0e-7 * V_LIQ,
    },
    V_L=V_LIQ,
    T_K=T_K,
)

print("Initial liquid inventory (mol):")
for sp, n in liquid.n_mol.items():
    print(f"  {sp:>14}: {n:.4e}")

## 9 · Declare the transfer models

Identical to Example2.

In [ ]:
transfer_models = {
    "O2":  KineticTransferModel(
               AD_BASIC.partition_models["O2"],
               k_transfer=KLA_PER_H["O2"],
           ),
    "CO2": KineticTransferModel(
               AD_BASIC.partition_models["CO2"],
               k_transfer=KLA_PER_H["CO2"],
               transfer_basis="molecular",
           ),
    "N2":  EquilibriumTransferModel(
               AD_BASIC.partition_models["N2"],
           ),
}

print("Transfer models:")
for sp, tm in transfer_models.items():
    print(f"  {sp:>4}: {tm}")

## 10 · Assemble the ControlVolume

Identical to Example2.

In [ ]:
cv = ControlVolume(
    phases          = {"gas": gas, "liquid": liquid},
    transfer_models = transfer_models,
    reaction_system = rxn_system,
    label           = "cstr",
)

print("CV label:", cv.label)
print("Phases:  ", list(cv.phases.keys()))
print("Transfer models:", list(cv.transfer_models.keys()))

## 11 · Attach boundaries

This section is where the CSTR diverges from the batch fermenter. In addition
to the gas sparging and pressure relief carried over from Example2, two new
liquid boundaries are added:

### Gas boundaries (unchanged from Example2)

- **`GasFeed`** — sparge air at 1 vvm.
- **`PressureReliefVent`** — instant pressure relief at 1.10 atm.

### Liquid boundaries (new in Example3)

- **`LiquidFeed`** — continuously adds fresh medium at flow rate $Q$. The
  feed contains only substrate (`AceticAcid` at `S_FEED_G_L`); no biomass is
  fed (free-cell operation). The molar inlet rate for species $i$ is:

  $$\dot{n}_{i,\text{in}} = C_{i,\text{feed}} \times Q$$

- **`LiquidDrain`** — continuously removes reactor broth at the same flow rate
  $Q$, maintaining a constant liquid volume. All species leave at their current
  reactor concentration (perfect mixing assumption):

  $$\dot{n}_{i,\text{out}} = C_{i,\text{reactor}} \times Q$$

  The drain uses an exponential formulation for numerical stability at any
  $D \times \Delta t$ product.

In [ ]:
# Gas boundaries — identical to Example2
cv.boundaries.append(GasFeed(
    vvm_min          = 1.0,
    y                = {"O2": 0.21, "N2": 0.79},
    P_inlet_atm      = 1.0,
    phase_key        = "gas",
    liquid_phase_key = "liquid",
    label            = "air_sparge",
))
cv.boundaries.append(PressureReliefVent(P_set_atm=1.10, mode="instant"))

print("Gas boundaries attached.")

In [ ]:
# Liquid feed — substrate enters at S_FEED_G_L, no biomass in feed
S_FEED_MOL_L = S_FEED_G_L / float(ACETIC_ACID.MW)

liquid_feed = LiquidFeed(
    Q_L_per_h       = Q_L_PER_H,
    feed_conc_mol_L = {ACETIC_ACID.id: S_FEED_MOL_L},
    phase_key       = "liquid",
    label           = "substrate_feed",
)
cv.boundaries.append(liquid_feed)

# Liquid drain — effluent exits at reactor concentration; maintains constant volume
liquid_drain = LiquidDrain(
    Q_L_per_h = Q_L_PER_H,
    phase_key = "liquid",
    label     = "effluent_drain",
)
cv.boundaries.append(liquid_drain)

print(f"LiquidFeed:  Q = {liquid_feed.Q_L_per_h:.4f} L/h, "
      f"C_AceticAcid = {S_FEED_MOL_L:.4f} mol/L ({S_FEED_G_L:.1f} g/L)")
print(f"LiquidDrain: Q = {liquid_drain.Q_L_per_h:.4f} L/h (removes all species)")

In [ ]:
# pH controller — identical to Example2
ph_ctrl = PHController(
    setpoint         = PH_SETPOINT,
    Kp               = 0.5,
    Ki               = 0.0,
    chemical_id      = "H3PO4",
    base_chemical_id = "NaOH",
    max_add_molL_hr  = 0.05,
)

print("All boundaries:", [b.label for b in cv.boundaries])
print("pH controller setpoint:", ph_ctrl.setpoint)

## 12 · Build the Simulation

In [ ]:
sim = Simulation(
    cvs         = {"main": cv},
    controllers = [ph_ctrl],
    label       = "cstr_sim",
)

print("Simulation:", sim.label)
print("CVs:       ", list(sim.cvs.keys()))

## 13 · Run

The simulation runs for 10 h = 2 HRTs. By the end of the run the reactor
should be within a few percent of the theoretical steady state.

In [ ]:
result = sim.run(tau_h=TAU_H, n_steps=N_STEPS)
print(f"Finished in {result.runtime_s:.2f} s")

## 14 · Theoretical steady state

The Monod CSTR steady state has a closed-form solution. Below we compute it
and compare it to the simulation endpoint. Good agreement validates that:
1. The `LiquidFeed` and `LiquidDrain` rates are correctly balanced.
2. The simulation ran long enough for transients to decay.

In [ ]:
MW_S = float(ACETIC_ACID.MW)
MW_X = float(YEAST.MW)

# Theoretical steady state (Monod CSTR, no maintenance or death term)
S_ss_theory = KS_G_L * D_PER_H / (MU_MAX - D_PER_H)   # g/L
X_ss_theory = YIELD  * (S_FEED_G_L - S_ss_theory)       # g/L

print(f"Theoretical steady state (Monod CSTR):")
print(f"  Residual substrate S* = {S_ss_theory:.4f} g/L")
print(f"  Biomass            X* = {X_ss_theory:.4f} g/L")
print(f"  Net growth = D = {D_PER_H:.3f} /h  (μ_max × S*/(Ks+S*) check: "
      f"{MU_MAX * S_ss_theory / (KS_G_L + S_ss_theory):.3f} /h)")

# Simulation result at the final timestep
liq = result.liquid_mol["main"]
C_S_final = liq["AceticAcid"][-1] / V_LIQ * MW_S   # g/L
C_X_final = liq["Yeast"][-1]      / V_LIQ * MW_X   # g/L

print(f"\nSimulation endpoint (t = {TAU_H:.0f} h):")
print(f"  Substrate S  = {C_S_final:.4f} g/L  "
      f"(theory: {S_ss_theory:.4f}, error: {abs(C_S_final - S_ss_theory)/max(S_ss_theory,1e-9)*100:.1f}%)")
print(f"  Biomass   X  = {C_X_final:.4f} g/L  "
      f"(theory: {X_ss_theory:.4f}, error: {abs(C_X_final - X_ss_theory)/X_ss_theory*100:.1f}%)")

## 15 · Results summary

In [ ]:
cv_key = "main"

print(f"t = 0 to {TAU_H} h  ({N_STEPS} steps)  |  D = {D_PER_H} /h  |  pH setpoint = {PH_SETPOINT}")
print("\nInitial -> Final liquid inventory (mol):")
for sp in sorted(result.liquid_mol[cv_key]):
    n0 = result.liquid_mol[cv_key][sp][0]
    nf = result.liquid_mol[cv_key][sp][-1]
    print(f"  {sp:>14}: {n0:>10.4e}  ->  {nf:>10.4e}")

pH = result.pH[cv_key]
pH_valid = pH[np.isfinite(pH)]
print(f"\npH: {pH_valid[0]:.3f}  ->  {pH_valid[-1]:.3f}")
print(f"Runtime: {result.runtime_s:.3f} s")

## 16 · Time-series plots

Dashed horizontal lines show the theoretical Monod steady state. The
simulation traces should converge to these values by the end of the run.

- **Substrate** (top left): starts at the initial 1.2 g/L batch charge, then
  quickly adjusts towards the very low CSTR steady state (feed dilutes the
  initial inventory; growth consumes it).
- **Biomass** (top right): rises from the 0.1 g/L inoculum as the high-yield
  feed sustains growth faster than the drain removes cells.
- **Dissolved O₂** (bottom left): tracks the balance between sparging supply
  and biological demand.
- **pH** (bottom right): held near the setpoint by the controller.

In [ ]:
cv_key = "main"
t   = result.t_h
liq = result.liquid_mol[cv_key]
pH  = result.pH[cv_key]

MW_S = float(ACETIC_ACID.MW)
MW_X = float(YEAST.MW)

# Convert mol → g/L
C_S  = liq["AceticAcid"] / V_LIQ * MW_S
C_X  = liq["Yeast"]      / V_LIQ * MW_X
C_O2 = liq.get("O2", np.zeros_like(t))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f"CSTR — D = {D_PER_H:.2f} /h, HRT = {1/D_PER_H:.0f} h, S_feed = {S_FEED_G_L:.1f} g/L",
             fontsize=13)

# Substrate
ax = axes[0, 0]
ax.plot(t, C_S, color="tab:orange", label="Simulation")
ax.axhline(S_ss_theory, ls="--", color="gray", lw=1.5, label=f"SS theory ({S_ss_theory:.3f} g/L)")
ax.set(xlabel="Time (h)", ylabel="Concentration (g/L)", title="Substrate (AceticAcid)")
ax.legend(fontsize=9)

# Biomass
ax = axes[0, 1]
ax.plot(t, C_X, color="tab:green", label="Simulation")
ax.axhline(X_ss_theory, ls="--", color="gray", lw=1.5, label=f"SS theory ({X_ss_theory:.3f} g/L)")
ax.set(xlabel="Time (h)", ylabel="Concentration (g/L)", title="Biomass (Yeast)")
ax.legend(fontsize=9)

# Dissolved O2
ax = axes[1, 0]
ax.plot(t, C_O2, color="tab:blue")
ax.set(xlabel="Time (h)", ylabel="mol", title="Dissolved O\u2082 (liquid)")

# pH
ax = axes[1, 1]
ax.plot(t, pH, color="tab:red", label="pH")
ax.axhline(PH_SETPOINT, ls="--", color="gray", label=f"setpoint ({PH_SETPOINT})")
ax.set(xlabel="Time (h)", ylabel="pH", title="pH")
ax.legend()

plt.tight_layout()
plt.show()